# Notebook for testing the geometry extraction pipeline on a few sample elements

## 1. Loading and Exploring the IFC Model

In [2]:
# imports
import sys
import ifcopenshell
import pandas as pd
import random

# helpers
sys.path.insert(0, "../../")
from geometric_extraction_helper import (
    _get_settings, 
    _get_geometry,
    extract_general_features,
    extract_aabb_features,
    extract_tfbb_features,
    extract_topology_features,
    extract_material_features,
    extract_ray_features,
    extract_all_features,
    iter_elements_with_features,
    _IGNORE_TYPES
    )

# set seed for reproducibility
from _settings import SEED
random.seed(SEED)

# sett global settings for geometry extraction
SETTINGS = _get_settings()

In [2]:
IFC_PATH = "../../0_data/1_ifc_models/ZUST/ZUST_A_ARC_EBK_.ifc"

model = ifcopenshell.open(IFC_PATH)
print(f"Schema: {model.schema}")
print(f"Anzahl IfcProduct-Elemente: {len(model.by_type('IfcProduct'))}")

Schema: IFC4
Anzahl IfcProduct-Elemente: 3865


## 2. Testing single methods on a random element

In [3]:
# get manual picked element for testing
element = model.by_guid("2wcWhZx7r0DwHef36SJubU")
verts, faces = _get_geometry(element, SETTINGS)

print(f"GlobalId : {element.GlobalId}")
print(f"Name     : {element.Name}")
print(f"Typ      : {element.is_a()}")
print()

GlobalId : 2wcWhZx7r0DwHef36SJubU
Name     : 2Decke-001
Typ      : IfcSlab



In [4]:
general = extract_general_features(verts, faces)
general

{'geom_volume': 68.76552489123205,
 'geom_surface_area': 577.5429992192228,
 'geom_projected_area': 310.5486001524407,
 'geom_centroid_x': 5.623352154377207,
 'geom_centroid_y': -13.122356798148795,
 'geom_centroid_z': -0.12501782205019368,
 'geom_z_min': -0.2499999999999982,
 'geom_z_max': 0.0,
 'geom_z_range': 0.2499999999999982,
 'geom_ratio_area_vol': 8.398728870792963,
 'geom_compactness': 0.14054443119427573,
 'geom_layer_count': 0}

In [5]:
# extract geometry and compute AABB features
aabb   = extract_aabb_features(verts)
aabb

{'aabb_min_x': 0.0,
 'aabb_min_y': -26.34,
 'aabb_min_z': -0.2499999999999982,
 'aabb_max_x': 11.79000000732167,
 'aabb_max_y': 0.0,
 'aabb_max_z': 0.0,
 'aabb_len_x': 11.79000000732167,
 'aabb_len_y': 26.34,
 'aabb_len_z': 0.2499999999999982,
 'aabb_ratio_z_x': np.float64(0.02120441050421938),
 'aabb_ratio_z_y': np.float64(0.009491268033409194),
 'aabb_ratio_x_y': np.float64(0.44760820073354857),
 'aabb_diagonal': 28.859352040069176,
 'aabb_volume': 77.63715004821263}

In [6]:
# extra TFBB features
tfbb = extract_tfbb_features(verts)
tfbb

{'tfbb_extent_0': 26.970610794977468,
 'tfbb_extent_1': 13.27654759728239,
 'tfbb_extent_2': 0.29948169939828784,
 'tfbb_volume': 107.23738806801384,
 'tfbb_ratio_01': np.float64(2.031447603177964),
 'tfbb_ratio_02': np.float64(90.05762572192637),
 'tfbb_ratio_12': np.float64(44.33174923194753),
 'tfbb_linearity': 0.5331543220940452,
 'tfbb_planarity': 0.46650776657822823,
 'tfbb_sphericity': 0.0003379113277266268,
 'tfbb_primary_ax_x': -0.05713487866868866,
 'tfbb_primary_ax_y': -0.9983664660514969,
 'tfbb_primary_ax_z': 7.143780402214609e-05}

In [7]:
# extra topology features
topo = extract_topology_features(verts, faces)
topo

{'topo_vertex_count': 113.0,
 'topo_face_count': 230.0,
 'topo_unique_edge_count': 345.0,
 'topo_euler_characteristic': -2.0,
 'topo_genus': 2.0,
 'topo_max_face_area': 48.71814880371354,
 'topo_avg_face_area': 2.511056518344447,
 'topo_vertex_edge_ratio': 0.32753623188405795,
 'topo_connected_components': 1.0}

In [8]:
# extra material features
materials = extract_material_features(element)
materials

{'mat_allgemein': 0,
 'mat_aluminium': 0,
 'mat_backstein': 0,
 'mat_bekleidung': 0,
 'mat_belag': 0,
 'mat_beton': 1,
 'mat_dämm': 0,
 'mat_foamglas': 0,
 'mat_gips': 0,
 'mat_glas': 0,
 'mat_holz': 0,
 'mat_werkstoff': 0,
 'mat_kalksandstein': 0,
 'mat_keramik': 0,
 'mat_kies': 0,
 'mat_kunststein': 0,
 'mat_kunststoff': 0,
 'mat_metall': 0,
 'mat_mörtel': 0,
 'mat_naturstein': 0,
 'mat_putz': 0,
 'mat_stahl': 1,
 'mat_zement': 0}

In [11]:
# ray casting ist nested in methods and will be testet below due to computational reductions

## 4. Test it on a ifc file

In [9]:
# test main method
test_features = list()
max_rows = 5
counter = 0
for elem, feats in iter_elements_with_features(model):
    test_features.append(feats)
    print(elem.GlobalId, feats["geom_volume"])
    counter += 1
    if counter >= max_rows:
        break

feats.keys()

3E7FZ3G714ZfWq$HpvWPPe 0.013999999999999667
0VH0$VM1H5ahhbSV8YO9Id 0.026375999999999983
3rvBwfBqP4tgvk5ppmELEf 0.026375999999999983
26slolJhv9wueyE4nRGaWv 0.026375999999999983
1H6Kfo9W56NRZ3Og7N6NDO 0.06822899961979662


dict_keys(['aabb_min_x', 'aabb_min_y', 'aabb_min_z', 'aabb_max_x', 'aabb_max_y', 'aabb_max_z', 'aabb_len_x', 'aabb_len_y', 'aabb_len_z', 'aabb_ratio_z_x', 'aabb_ratio_z_y', 'aabb_ratio_x_y', 'aabb_diagonal', 'aabb_volume', 'geom_volume', 'geom_surface_area', 'geom_projected_area', 'geom_centroid_x', 'geom_centroid_y', 'geom_centroid_z', 'geom_z_min', 'geom_z_max', 'geom_z_range', 'geom_ratio_area_vol', 'geom_compactness', 'geom_layer_count', 'tfbb_extent_0', 'tfbb_extent_1', 'tfbb_extent_2', 'tfbb_volume', 'tfbb_ratio_01', 'tfbb_ratio_02', 'tfbb_ratio_12', 'tfbb_linearity', 'tfbb_planarity', 'tfbb_sphericity', 'tfbb_primary_ax_x', 'tfbb_primary_ax_y', 'tfbb_primary_ax_z', 'topo_vertex_count', 'topo_face_count', 'topo_unique_edge_count', 'topo_euler_characteristic', 'topo_genus', 'topo_max_face_area', 'topo_avg_face_area', 'topo_vertex_edge_ratio', 'topo_connected_components', 'mat_allgemein', 'mat_aluminium', 'mat_backstein', 'mat_bekleidung', 'mat_belag', 'mat_beton', 'mat_dämm', 'mat

In [10]:
pd.DataFrame(test_features)

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,mat_kunststein,mat_kunststoff,mat_metall,mat_mörtel,mat_naturstein,mat_putz,mat_stahl,mat_zement,horizontal_elements_above,horizontal_elements_below
0,0.0,0.0,-0.07,0.20,1.00,0.00,0.20,1.00,0.07,0.350000,...,0,0,0,0,0,0,0,0,77.0,0.0
1,-0.4,0.0,3.34,0.00,0.35,3.84,0.40,0.35,0.50,1.250000,...,0,0,0,0,0,0,0,0,NaN,NaN
2,-0.4,0.0,3.34,0.00,0.35,3.84,0.40,0.35,0.50,1.250000,...,0,0,0,0,0,0,0,0,NaN,NaN
3,-0.4,0.0,3.34,0.00,0.35,3.84,0.40,0.35,0.50,1.250000,...,0,0,0,0,0,0,0,0,NaN,NaN
4,0.0,0.0,-0.09,0.21,3.61,0.00,0.21,3.61,0.09,0.428571,...,0,0,0,0,0,0,0,0,50.0,16.0
